# Cross-encoders vs bi-encoders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Fast separate encoding versus slower joint interaction
The examples compute bi-encoder cosine, let cross-encoder scores revise the order, quantify shortlist savings, show recall dependence on cutoff, and visualize the two-stage pipeline.

In [ ]:
# 1) Bi-encoder retrieval uses precomputed document vectors
q=np.array([1.,0.]); docs=np.array([[.9,.1],[.7,.7],[0.,1.]])
bi=np.array([cos(q,d) for d in docs])
plt.figure(figsize=(4,4)); plt.quiver([0]*4,[0]*4,[q[0],*docs[:,0]],[q[1],*docs[:,1]],angles='xy',scale_units='xy',scale=1); plt.xlim(-.1,1); plt.ylim(-.1,1); plt.title('bi-encoder vector comparison')
assert np.allclose(bi,[0.9938837347,0.7071067812,0.0])

In [ ]:
# 2) Cross-encoder pair scores can revise the order
cross=np.array([0.62,0.88,0.10])
plt.figure(figsize=(6,3)); plt.bar(np.arange(3)-.15,bi,width=.3,label='bi'); plt.bar(np.arange(3)+.15,cross,width=.3,label='cross'); plt.xticks(range(3),['d1','d2','d3']); plt.legend(); plt.title('joint scoring lifts d2')
assert int(np.argmax(bi))==0 and int(np.argmax(cross))==1

In [ ]:
# 3) Shortlisting makes cross-encoder passes affordable
N=1_000_000; shortlist=np.array([10,100,1000]); passes=shortlist
plt.figure(figsize=(6,3)); plt.loglog(shortlist,passes,'-o'); plt.axhline(N,ls='--',c='r',label='score all'); plt.legend(); plt.xlabel('shortlist size'); plt.ylabel('cross passes'); plt.title('top-100 is 10,000x fewer pair passes')
assert N/100==10000

In [ ]:
# 4) Too-small bi-encoder cutoff can hide the cross-encoder winner
bi_order=list(np.argsort(bi)[::-1]); cross_best=int(np.argmax(cross)); visible_at_1=cross_best in bi_order[:1]; visible_at_2=cross_best in bi_order[:2]
plt.figure(figsize=(6,3)); plt.bar(['top-1 shortlist','top-2 shortlist'],[visible_at_1,visible_at_2]); plt.ylim(0,1.1); plt.title('candidate recall gates re-ranking')
assert not visible_at_1 and visible_at_2

In [ ]:
# 5) Two-stage score: retrieve many cheaply, re-rank few carefully
stage_cost=np.array([3*2,2])  # toy: 3 docs * dim 2 cosine coords, then 2 cross passes
plt.figure(figsize=(6,3)); plt.bar(['bi retrieval coord ops','cross passes on top-2'],stage_cost); plt.title('staged retrieval spends care only on candidates')
assert stage_cost[0]==6 and stage_cost[1]==2